In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/Users/prash/Projects/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7411.22it/s]


In [35]:
x = embeddings.embed_query("hello world")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import os 
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/Users/prash/Projects/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7628.24it/s]


In [ ]:
from langchain_pinecone import PineconeVectorStore

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=os.environ.get("PINECONE_KEY"))
index_name = "ragagent" 

if index_name not in pc.list_indexes():
    pc.create_index(
        name=index_name,
        vector_type="dense",
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
        deletion_protection="disabled",
        tags={
            "environment": "development"
        }
    )


In [21]:
index = pc.Index(index_name)

In [22]:
vector_store = PineconeVectorStore(embedding=embeddings, index=index, namespace="default")

In [23]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid
loader = PyPDFDirectoryLoader("../data/pdf") 

# Define text processing rules
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=40)



BATCH_SIZE = 50 
chunk_accumulator = []

for page in loader.lazy_load():
    # add id to object 
    page_chunks = text_splitter.split_documents([page])

    if len(chunk_accumulator) >= BATCH_SIZE:
        vector_store.add_documents(chunk_accumulator)
        chunk_accumulator = []
    
    chunk_accumulator.extend(page_chunks)

vector_store.add_documents(chunk_accumulator)
        


['c1dd34a6-2b29-4f1a-a10e-3db84e503050',
 '8c380149-2a21-44aa-a41e-217beb330828',
 '0248696d-8118-4030-82cc-4f4bdc98d418',
 '17623025-d801-4ad3-afde-0eec1bb57c2e',
 '4913fd7b-4824-49c7-a8ec-841f357b079c',
 '316579ce-3c1f-4e9c-b456-38aa39091fa8']